# TIỀN XỬ LÝ DỮ LIỆU VÀ KỸ THUẬT ĐẶC TRƯNG
## Đồ án môn học: Dự đoán nhu cầu xe đạp công cộng (UCI Bike Sharing Dataset)

Sổ tay này thực hiện các bước làm sạch dữ liệu, xử lý giá trị bất thường, loại bỏ các cột gây rò rỉ dữ liệu và đa cộng tuyến, chuẩn bị tập dữ liệu hoàn chỉnh để huấn luyện mô hình.


# 1. Khởi tạo môi trường và đọc dữ liệu


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Đọc tập dữ liệu theo giờ gốc
df = pd.read_csv('../data/hour.csv')
print(f"Kích thước tập dữ liệu gốc: {df.shape[0]} dòng, {df.shape[1]} cột")
df.head()

Kích thước tập dữ liệu gốc: 17379 dòng, 17 cột


# 2. Loại bỏ các cột định danh và chống rò rỉ dữ liệu (Data Leakage)

Dựa trên kết quả phân tích dữ liệu (EDA), nhóm em thực hiện loại bỏ:
- **`instant` và `dteday`:** Chỉ là số thứ tự và ngày ghi nhận dạng chuỗi, không mang tính quy luật để mô hình học nên nhóm em xóa đi.
- **`casual` và `registered`:** Vì tổng của hai cột này chính bằng biến mục tiêu ($cnt = casual + registered$). Nếu giữ lại, mô hình sẽ bị rò rỉ dữ liệu (Data Leakage), chỉ cần học phép cộng là đạt điểm tuyệt đối giả tạo mà không có giá trị dự đoán trong thực tế.


In [5]:
# Loại bỏ các cột định danh và các cột gây rò rỉ dữ liệu
drop_leakage_cols = ['instant', 'dteday', 'casual', 'registered']
df = df.drop(columns=drop_leakage_cols, errors='ignore')

print(f"Đã loại bỏ các cột: {drop_leakage_cols}")
print(f"Kích thước dữ liệu hiện tại: {df.shape}")

Đã loại bỏ các cột: ['instant', 'dteday', 'casual', 'registered']
Kích thước dữ liệu hiện tại: (17379, 13)


# 3. Loại bỏ đặc trưng đa cộng tuyến (atemp)

Ở bước khám phá dữ liệu, nhóm em thấy tương quan giữa `temp` (nhiệt độ thực tế) và `atemp` (nhiệt độ cảm nhận) lên tới xấp xỉ 0.99. Để tránh hiện tượng đa cộng tuyến làm sai lệch các hệ số hồi quy, nhóm em chỉ giữ lại biến đo lường khách quan là `temp` và xóa cột `atemp`.


In [7]:
# Loại bỏ cột atemp để triệt tiêu hiện tượng đa cộng tuyến
df = df.drop(columns=['atemp'], errors='ignore')

print("Đã loại bỏ cột 'atemp'.")
print(f"Kích thước dữ liệu hiện tại: {df.shape}")

Đã loại bỏ cột 'atemp'.
Kích thước dữ liệu hiện tại: (17379, 12)


# 4. Xử lý 22 dòng có độ ẩm bằng 0 (hum == 0)

Do cảm biến thời tiết bị trục trặc vào ngày 10/03/2011 nên có 22 dòng ghi nhận độ ẩm $0.00$. Để giữ trọn vẹn chuỗi thời gian liên tục 17,379 giờ, nhóm em thay thế các giá trị 0 này bằng giá trị **trung vị (Median)** của độ ẩm theo từng tháng và từng khung giờ tương ứng từ các ngày bình thường.


In [9]:
# Kiểm tra số lượng bản ghi hum == 0 trước khi xử lý
zero_hum_count = (df['hum'] == 0).sum()
print(f"Số dòng có độ ẩm hum == 0: {zero_hum_count}")

if zero_hum_count > 0:
    # Tính giá trị trung vị độ ẩm theo (tháng, giờ) cho các bản ghi hợp lệ
    valid_hum = df[df['hum'] > 0]
    median_hum_by_mnth_hr = valid_hum.groupby(['mnth', 'hr'])['hum'].median()

    # Thay thế các giá trị 0 bằng trung vị theo tháng và khung giờ tương ứng
    df['hum'] = df.apply(
        lambda row: median_hum_by_mnth_hr.get((row['mnth'], row['hr']), 0.59) if row['hum'] == 0 else row['hum'],
        axis=1
    )

print(f"Số dòng có độ ẩm hum == 0 sau khi xử lý: {(df['hum'] == 0).sum()}")
print(f"Giá trị độ ẩm nhỏ nhất hiện tại: {df['hum'].min():.4f}")

Số dòng có độ ẩm hum == 0: 22
Số dòng có độ ẩm hum == 0 sau khi xử lý: 0
Giá trị độ ẩm nhỏ nhất hiện tại: 0.0800


# 5. Kiểm tra lại tập dữ liệu sau bước làm sạch cơ bản

Sau khi làm sạch, tập dữ liệu còn lại đúng 12 cột (11 đặc trưng độc lập và 1 biến mục tiêu `cnt`), không còn giá trị rỗng hay bất hợp lý, sẵn sàng cho các bước tiếp theo.


In [11]:
# Kiểm tra tổng quan kiểu dữ liệu và giá trị khuyết
print("=== THÔNG TIN DỮ LIỆU HIỆN TẠI ===")
df.info()
print("\n=== THỐNG KÊ MÔ TẢ CÁC BIẾN SỐ HỌC ===")
df[['temp', 'hum', 'windspeed', 'cnt']].describe().T

=== THÔNG TIN DỮ LIỆU HIỆN TẠI ===
<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   season      17379 non-null  int64  
 1   yr          17379 non-null  int64  
 2   mnth        17379 non-null  int64  
 3   hr          17379 non-null  int64  
 4   holiday     17379 non-null  int64  
 5   weekday     17379 non-null  int64  
 6   workingday  17379 non-null  int64  
 7   weathersit  17379 non-null  int64  
 8   temp        17379 non-null  float64
 9   hum         17379 non-null  float64
 10  windspeed   17379 non-null  float64
 11  cnt         17379 non-null  int64  
dtypes: float64(3), int64(9)
memory usage: 1.6 MB

=== THỐNG KÊ MÔ TẢ CÁC BIẾN SỐ HỌC ===


# 6. Xử lý ngoại lai bằng phương pháp IQR (Outlier Handling)

Ở bước khám phá dữ liệu, nhóm em đã xác định có 478 điểm ngoại lai của biến mục tiêu `cnt` theo tiêu chuẩn $Q_3 + 1.5 \times IQR$ theo từng mùa (chiếm 2.75% dữ liệu):
- **Bản chất ngoại lai:** Các điểm này phần lớn là giờ cao điểm tan tầm vào những ngày thời tiết đẹp. Tuy nhiên, các giá trị cực lớn (vọt lên gần 1.000 xe) sẽ tạo ra sai số bình phương khổng lồ $(y - \hat{y})^2$, khiến các mô hình như OLS Linear Regression và AdaBoost (vốn tự động tăng trọng số mẫu lỗi) bị hiện tượng học vẹt (overfitting).
- **Giải pháp xử lý:** Để vừa bảo vệ mô hình khỏi bị kéo lệch trọng số, vừa giữ trọn vẹn 17.379 giờ quan sát chuỗi thời gian liên tục, nhóm em áp dụng kỹ thuật **Capping (Winsorization)**: giới hạn trần các điểm vượt ngưỡng về đúng giá trị râu trên của từng mùa tương ứng (thay vì xóa bỏ làm mất dữ liệu đỉnh).
- **Đối với các biến đầu vào:** Nhóm biến thời gian và phân loại rời rạc có miền giá trị cố định nên không áp dụng ngoại lai thống kê; nhóm biến thời tiết (`temp`, `hum`, `windspeed`) đã được kiểm tra (xử lý xong 22 dòng lỗi cảm biến độ ẩm) và giữ nguyên biến thiên tự nhiên để chuẩn hóa ở bước sau.


In [13]:
# Áp dụng Capping (Winsorization) theo ngưỡng IQR cho biến mục tiêu cnt
df_clean = df.copy()
# Chuyển cnt sang float để có thể gán giá trị ngưỡng râu trên dạng thập phân
df_clean['cnt'] = df_clean['cnt'].astype(float)

season_col = 'season' if 'season' in df_clean.columns else None
if season_col:
    for season in df_clean['season'].unique():
        sub_idx = df_clean[df_clean['season'] == season].index
        q1 = df_clean.loc[sub_idx, 'cnt'].quantile(0.25)
        q3 = df_clean.loc[sub_idx, 'cnt'].quantile(0.75)
        iqr = q3 - q1
        upper_limit = q3 + 1.5 * iqr
        mask = (df_clean['season'] == season) & (df_clean['cnt'] > upper_limit)
        df_clean.loc[mask, 'cnt'] = upper_limit
else:
    q1 = df_clean['cnt'].quantile(0.25)
    q3 = df_clean['cnt'].quantile(0.75)
    iqr = q3 - q1
    upper_limit = q3 + 1.5 * iqr
    df_clean['cnt'] = df_clean['cnt'].clip(upper=upper_limit)

print(f"Giá trị cnt lớn nhất trước khi xử lý ngoại lai: {df['cnt'].max():.1f}")
print(f"Giá trị cnt lớn nhất sau khi Capping IQR: {df_clean['cnt'].max():.1f}")
df = df_clean


Giá trị cnt lớn nhất trước khi xử lý ngoại lai: 977.0
Giá trị cnt lớn nhất sau khi Capping IQR: 760.5


# 7. Kỹ thuật tạo đặc trưng chu kỳ thời gian (Cyclical Feature Encoding)

Các biến thời gian như giờ trong ngày (`hr`: 0–23), tháng (`mnth`: 1–12), và ngày trong tuần (`weekday`: 0–6) có bản chất là chu kỳ tuần hoàn khép kín. Ví dụ: 23h đêm và 0h sáng thực tế chỉ cách nhau 1 giờ, nhưng nếu để dạng số nguyên tuyến tính thì các mô hình (như hồi quy tuyến tính, kNN) sẽ hiểu là chúng cách xa nhau 23 đơn vị.

Để mô hình hiểu được tính liên tục vòng tròn, nhóm em sử dụng hàm lượng giác $\sin$ và $\cos$ để ánh xạ các biến này lên đường tròn đơn vị:
$$x_{\sin} = \sin\left(\frac{2\pi \cdot t}{T}\right), \quad x_{\cos} = \cos\left(\frac{2\pi \cdot t}{T}\right)$$
với $T = 24$ (cho giờ), $T = 12$ (cho tháng), và $T = 7$ (cho thứ).

In [15]:
# Mã hóa chu kỳ lượng giác sin/cos cho giờ (24), tháng (12) và thứ (7)
df['hr_sin'] = np.sin(2 * np.pi * df['hr'] / 24)
df['hr_cos'] = np.cos(2 * np.pi * df['hr'] / 24)

df['mnth_sin'] = np.sin(2 * np.pi * df['mnth'] / 12)
df['mnth_cos'] = np.cos(2 * np.pi * df['mnth'] / 12)

df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)

print("Đã tạo 6 đặc trưng chu kỳ lượng giác: hr_sin, hr_cos, mnth_sin, mnth_cos, weekday_sin, weekday_cos")
df[['hr', 'hr_sin', 'hr_cos', 'mnth', 'mnth_sin', 'mnth_cos']].head()

Đã tạo 6 đặc trưng chu kỳ lượng giác: hr_sin, hr_cos, mnth_sin, mnth_cos, weekday_sin, weekday_cos


# 8. Tạo đặc trưng mới: Giờ cao điểm (is_rush_hour)

Dựa trên kết quả phân tích biểu đồ Fever chart ở bước EDA, các đỉnh nhu cầu cao nhất chỉ bùng nổ vào các ngày làm việc bình thường (`workingday == 1`) tại khung giờ sáng (7h, 8h, 9h) và chiều tối (17h, 18h, 19h). 

Nhóm em tạo biến nhị phân `is_rush_hour` để giúp các thuật toán nắm bắt trực tiếp tín hiệu giờ cao điểm.

In [17]:
# Tạo đặc trưng nhị phân is_rush_hour
df['is_rush_hour'] = ((df['workingday'] == 1) & (df['hr'].isin([7, 8, 9, 17, 18, 19]))).astype(int)

rush_count = df['is_rush_hour'].sum()
print(f"Số giờ cao điểm (is_rush_hour = 1): {rush_count} / {len(df)} giờ (chiếm {rush_count/len(df)*100:.1f}%)")
print(f"Lượng xe thuê trung bình giờ cao điểm: {df[df['is_rush_hour'] == 1]['cnt'].mean():.1f} xe/giờ")
print(f"Lượng xe thuê trung bình giờ bình thường: {df[df['is_rush_hour'] == 0]['cnt'].mean():.1f} xe/giờ")

Số giờ cao điểm (is_rush_hour = 1): 2983 / 17379 giờ (chiếm 17.2%)
Lượng xe thuê trung bình giờ cao điểm: 381.4 xe/giờ
Lượng xe thuê trung bình giờ bình thường: 146.2 xe/giờ


# 9. Mã hóa biến phân loại bằng One-Hot Encoding

Biến `season` (4 mùa) và `weathersit` (4 loại thời tiết) là các biến danh nghĩa (không có quan hệ thứ tự hơn kém định lượng). Để mô hình không bị hiểu nhầm giá trị $1, 2, 3, 4$ là cấp bậc tuyến tính, nhóm em áp dụng mã hóa One-Hot với tham số `drop_first=True` để tránh hiện tượng bẫy biến giả (Dummy Variable Trap).

In [19]:
# Áp dụng One-Hot Encoding cho season và weathersit nếu chưa mã hóa
cat_cols = [c for c in ['season', 'weathersit'] if c in df.columns]
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

print(f"Kích thước dữ liệu hiện tại: {df.shape[0]} dòng, {df.shape[1]} cột")
print("Danh sách các thuộc tính:")
print(list(df.columns))

Kích thước dữ liệu hiện tại: 17379 dòng, 23 cột
Danh sách các thuộc tính:
['yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'temp', 'hum', 'windspeed', 'cnt', 'hr_sin', 'hr_cos', 'mnth_sin', 'mnth_cos', 'weekday_sin', 'weekday_cos', 'is_rush_hour', 'season_2', 'season_3', 'season_4', 'weathersit_2', 'weathersit_3', 'weathersit_4']


# 10. Đánh giá và Chọn lọc đặc trưng bằng Kiểm định Thống kê (Feature Selection)

Để đánh giá khách quan mức độ đóng góp của từng đặc trưng lên biến mục tiêu `cnt`, nhóm em áp dụng đồng thời hai phương pháp kiểm định thống kê:
1. **ANOVA F-test (`f_regression`):** Đo lường mối quan hệ tuyến tính giữa biến đầu vào và biến mục tiêu.
2. **Thông tin tương hỗ Mutual Information (`mutual_info_regression`):** Đo lường mức độ phụ thuộc phi tuyến dựa trên lý thuyết thông tin Shannon.

In [21]:
from sklearn.feature_selection import f_regression, mutual_info_regression

# Tách tập đặc trưng X và biến mục tiêu y
X = df.drop(columns=['cnt'])
y = df['cnt']

# Tính toán F-Score và Mutual Information
f_scores, p_values = f_regression(X, y)
mi_scores = mutual_info_regression(X, y, random_state=42)

df_importance = pd.DataFrame({
    'Feature': X.columns,
    'F_Score': f_scores,
    'Mutual_Info': mi_scores
})

# Vẽ biểu đồ cột ngang so sánh 2 tiêu chuẩn
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
fig.suptitle('Xếp hạng Mức độ Ảnh hưởng của Đặc trưng lên Nhu cầu Thuê xe\n(ANOVA F-test vs. Mutual Information)', 
             fontsize=14, fontweight='bold', y=1.02)

# 1. Panel F-Score (Tuyến tính)
df_f = df_importance.sort_values('F_Score', ascending=True)
colors_f = ['#e74c3c' if f == 'is_rush_hour' else '#3498db' for f in df_f['Feature']]
axes[0].barh(df_f['Feature'], df_f['F_Score'], color=colors_f, edgecolor='black', alpha=0.85)
axes[0].set_title('1. Kiểm định ANOVA F-test (Quan hệ tuyến tính)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Giá trị F-Score', fontsize=11)
axes[0].grid(axis='x', linestyle='--', alpha=0.5)

# 2. Panel Mutual Information (Phi tuyến)
df_mi = df_importance.sort_values('Mutual_Info', ascending=True)
colors_mi = ['#e74c3c' if f == 'is_rush_hour' else '#2ecc71' for f in df_mi['Feature']]
axes[1].barh(df_mi['Feature'], df_mi['Mutual_Info'], color=colors_mi, edgecolor='black', alpha=0.85)
axes[1].set_title('2. Thông tin tương hỗ Mutual Information (Quan hệ phi tuyến)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hệ số Mutual Information (nats)', fontsize=11)
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../output/step2_prep_01_feature_selection.png', dpi=300, bbox_inches='tight')
plt.show()

**Nhận xét từ kết quả kiểm định:**
- Cả hai phương pháp đều chỉ ra rằng **khung giờ trong ngày (`hr`, `hr_sin`, `hr_cos`)** và **nhiệt độ (`temp`)** là những đặc trưng có mức độ chi phối mạnh nhất đến nhu cầu thuê xe.
- Đặc biệt, đặc trưng mới do nhóm em tạo là **`is_rush_hour` đạt F-Score cao nhất vượt trội (hơn 6.300)**, chứng minh việc tạo biến giờ cao điểm dựa trên khám phá thực tế mang lại tín hiệu tuyến tính cực kỳ mạnh mẽ cho mô hình.
- Các đặc trưng chu kỳ $\sin/\cos$ cũng nằm trong nhóm có Mutual Information cao, cho thấy hiệu quả rõ rệt của kỹ thuật mã hóa chu kỳ lượng giác.

# 11. Phân cụm K-Means và Trực quan hóa Không gian Đặc trưng bằng PCA

Nhằm làm giàu thêm thông tin cho các mô hình học máy, nhóm em áp dụng hai kỹ thuật học không giám sát (Unsupervised Learning):
1. **K-Means Clustering:** Phân tích hồ sơ nhu cầu thuê xe của 24 khung giờ (lượng thuê trung bình, lượng thuê ngày làm việc và ngày nghỉ/cuối tuần) để gom thành $k=3$ cụm hành vi đặc trưng: *Thấp điểm đêm/sáng sớm*, *Bình thường ban ngày & tối*, và *Cao điểm bùng nổ*. Đặc trưng mới `hour_cluster` được tạo ra để cung cấp tín hiệu nhóm hành vi cho mô hình.
2. **PCA (Principal Component Analysis):** Giảm chiều không gian đặc trưng đa biến về 2 thành phần chính (PC1 và PC2) để trực quan hóa sự phân bố và ranh giới phân tách của các cụm trong không gian 2 chiều.

In [24]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Xây dựng hồ sơ hành vi 24 khung giờ và chạy K-Means
hourly_stats = df.groupby("hr").agg(
    cnt_mean=("cnt", "mean"),
    cnt_workingday=("cnt", lambda x: df.loc[x.index][df.loc[x.index, "workingday"] == 1]["cnt"].mean()),
    cnt_weekend=("cnt", lambda x: df.loc[x.index][df.loc[x.index, "workingday"] == 0]["cnt"].mean())
).reset_index()

scaler_h = StandardScaler()
X_h_scaled = scaler_h.fit_transform(hourly_stats[["cnt_mean", "cnt_workingday", "cnt_weekend"]])

kmeans_h = KMeans(n_clusters=3, random_state=42, n_init=10)
hourly_stats["cluster"] = kmeans_h.fit_predict(X_h_scaled)

# Sắp xếp lại thứ tự nhãn cụm theo lượng xe tăng dần: 0 (Thấp), 1 (Trung bình), 2 (Cao điểm)
cluster_means = hourly_stats.groupby("cluster")["cnt_mean"].mean()
cluster_rank = cluster_means.sort_values().index
cluster_map = {old: new for new, old in enumerate(cluster_rank)}
hourly_stats["hour_cluster"] = hourly_stats["cluster"].map(cluster_map)

# Ánh xạ đặc trưng cụm mới hour_cluster vào tập dữ liệu chính
hr_to_cluster = dict(zip(hourly_stats["hr"], hourly_stats["hour_cluster"]))
df["hour_cluster"] = df["hr"].map(hr_to_cluster)

# 2. Áp dụng PCA giảm chiều không gian đặc trưng về 2D
feature_cols = [c for c in df.columns if c not in ["cnt", "hour_cluster", "pca_1", "pca_2"]]
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(df[feature_cols])

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df["pca_1"] = X_pca[:, 0]
df["pca_2"] = X_pca[:, 1]

# 3. Vẽ biểu đồ kết hợp Hồ sơ phân cụm và Không gian 2D của PCA
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cluster_names = {
    0: "Cụm 0: Đêm khuya & rạng sáng (Nhu cầu rất thấp)",
    1: "Cụm 1: Ban ngày sinh hoạt thông thường (Nhu cầu ổn định)",
    2: "Cụm 2: Cao điểm đi làm & tan tầm (Nhu cầu bùng nổ)"
}
colors = ["#3498db", "#f39c12", "#e74c3c"]

# Panel 1: Hồ sơ phân cụm 24 khung giờ
for c in range(3):
    sub = hourly_stats[hourly_stats["hour_cluster"] == c]
    axes[0].scatter(sub["hr"], sub["cnt_mean"], color=colors[c], s=120, zorder=5, label=cluster_names[c])

axes[0].plot(hourly_stats["hr"], hourly_stats["cnt_mean"], color="gray", linestyle="--", alpha=0.6, zorder=2)
axes[0].plot(hourly_stats["hr"], hourly_stats["cnt_workingday"], color="#2c3e50", linestyle="-", alpha=0.7, label="Lượng thuê ngày làm việc (workingday=1)", zorder=3)
axes[0].plot(hourly_stats["hr"], hourly_stats["cnt_weekend"], color="#16a085", linestyle=":", alpha=0.7, label="Lượng thuê cuối tuần/nghỉ (workingday=0)", zorder=3)
axes[0].set_title("1. Phân cụm K-Means cho 24 Khung giờ (k = 3)", fontsize=13, fontweight="bold", pad=10)
axes[0].set_xlabel("Khung giờ trong ngày (hr: 0 - 23)", fontsize=11)
axes[0].set_ylabel("Lượng xe thuê trung bình (cnt)", fontsize=11)
axes[0].set_xticks(range(0, 24))
axes[0].legend(loc="upper left", fontsize=9, frameon=True)
axes[0].grid(True, linestyle="--", alpha=0.5)

# Panel 2: Chiếu không gian đặc trưng lên 2D bằng PCA
df_sample = df.sample(n=3000, random_state=42)
for c in range(3):
    sub = df_sample[df_sample["hour_cluster"] == c]
    axes[1].scatter(sub["pca_1"], sub["pca_2"], color=colors[c], alpha=0.35, s=25, label=cluster_names[c], edgecolors="none")

# Vẽ riêng từng dấu X theo màu cụm (không kèm chữ để biểu đồ thoáng và không bị che)
centroids_pca = df.groupby("hour_cluster")[["pca_1", "pca_2"]].mean()
centroid_legend_names = ["Tâm Cụm 0 (Đêm)", "Tâm Cụm 1 (Ngày)", "Tâm Cụm 2 (Cao điểm)"]
for c in range(3):
    cx = centroids_pca.loc[c, "pca_1"]
    cy = centroids_pca.loc[c, "pca_2"]
    axes[1].scatter(cx, cy, color=colors[c], marker="X", s=220, edgecolors="black", linewidth=2.2, zorder=10, label=centroid_legend_names[c])

axes[1].set_title(f"2. Không gian đặc trưng 2D qua PCA (PC1: {pca.explained_variance_ratio_[0]*100:.1f}%, PC2: {pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=13, fontweight="bold", pad=10)
axes[1].set_xlabel(f"Thành phần chính 1 - PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% phương sai)", fontsize=11)
axes[1].set_ylabel(f"Thành phần chính 2 - PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% phương sai)", fontsize=11)
axes[1].legend(loc="lower right", fontsize=8.5, frameon=True)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("../output/step2_prep_02_pca_kmeans.png", dpi=300, bbox_inches="tight")
plt.show()

print("=== PHÂN BỐ CÁC CỤM KHUNG GIỜ (HOUR_CLUSTER) ===")
for c in range(3):
    count = (df["hour_cluster"] == c).sum()
    pct = count / len(df) * 100
    avg = df[df["hour_cluster"] == c]["cnt"].mean()
    cx = centroids_pca.loc[c, "pca_1"]
    cy = centroids_pca.loc[c, "pca_2"]
    print(f"{cluster_names[c]}: {count} quan sát ({pct:.1f}%), Lượng thuê TB: {avg:.1f} xe/giờ -> Tâm PCA: ({cx:.3f}, {cy:.3f})")
print(f"\nKích thước tập dữ liệu hiện tại: {df.shape[0]} dòng, {df.shape[1]} cột")

=== PHÂN BỐ CÁC CỤM KHUNG GIỜ (HOUR_CLUSTER) ===
Cụm 0: Đêm khuya & rạng sáng (Nhu cầu rất thấp): 6457 quan sát (37.2%), Lượng thuê TB: 49.7 xe/giờ -> Tâm PCA: (0.295, -0.362)
Cụm 1: Ban ngày sinh hoạt thông thường (Nhu cầu ổn định): 8009 quan sát (46.1%), Lượng thuê TB: 228.7 xe/giờ -> Tâm PCA: (-0.150, 0.175)
Cụm 2: Cao điểm đi làm & tan tầm (Nhu cầu bùng nổ): 2913 quan sát (16.8%), Lượng thuê TB: 374.1 xe/giờ -> Tâm PCA: (-0.240, 0.322)

Kích thước tập dữ liệu hiện tại: 17379 dòng, 26 cột


**Nhận xét từ kết quả phân cụm K-Means và phân tích PCA:**
- **Phân cụm K-Means ($k=3$):** Thuật toán tự động nhóm 24 khung giờ thành 3 mức độ nhu cầu rất trực quan theo đời sống thực tế:
  + *Cụm 0 (Đêm khuya & rạng sáng: 0h-6h, 22h-23h):* Gồm 9 khung giờ khi người dân ngủ nghỉ, nhu cầu chạm đáy chỉ 49.7 xe/giờ (chiếm 37.2% dữ liệu).
  + *Cụm 1 (Ban ngày sinh hoạt thông thường: 7h, 9h-16h, 20h-21h):* Gồm 11 khung giờ sinh hoạt và làm việc ban ngày với nhu cầu ổn định khoảng 228.7 xe/giờ (chiếm 46.1% dữ liệu).
  + *Cụm 2 (Cao điểm đi làm & tan tầm: 8h, 17h-19h):* Gồm 4 khung giờ đỉnh điểm của người đi làm và học sinh với lượng thuê bùng nổ lên tới 374.1 xe/giờ (chiếm 16.8% dữ liệu).
- **Phân tích không gian PCA:** Hai thành phần chính PC1 và PC2 giải thích tổng cộng **25.2% phương sai** của không gian đặc trưng đa biến. Trên mặt phẳng 2D, các điểm dữ liệu thuộc 3 cụm có sự phân tách rõ nét dọc theo trục tọa độ và các tâm cụm (Centroids) phân bố cách biệt, chứng minh đặc trưng `hour_cluster` mang lại giá trị phân loại hành vi cao cho các mô hình hồi quy ở giai đoạn sau.

# 12. Phân chia tập dữ liệu Huấn luyện & Kiểm tra và Chuẩn hóa đặc trưng (Train/Test Split & Feature Scaling)

Đây là bước chốt hạ của giai đoạn tiền xử lý, nhóm em thực hiện với các nguyên tắc học máy chặt chẽ:
1. **Nguyên tắc chống rò rỉ dữ liệu (Zero Data Leakage):**
   - Chia tập dữ liệu thành **80% Huấn luyện (Train)** và **20% Kiểm tra (Test)** với `random_state=42` **TRƯỚC** khi thực hiện bất kỳ bước chuẩn hóa nào.
   - Các bộ chuẩn hóa (`StandardScaler`, `MinMaxScaler`, `RobustScaler`) **CHỈ ĐƯỢC PHÉP `.fit()` trên `X_train`** để học tham số thống kê ($\mu, \sigma, \min, \max$). Sau đó, dùng chính tham số này để `.transform()` cho cả `X_train` và `X_test`.
2. **Lựa chọn các bộ chuẩn hóa:**
   - `StandardScaler`: Chuẩn hóa về phân phối $Z$-score (trung bình 0, độ lệch chuẩn 1), là bộ chuẩn hóa chính cho các mô hình hồi quy tuyến tính, Ridge, Lasso, SVR và MLP.
   - `MinMaxScaler` và `RobustScaler`: Được chuẩn bị sẵn sàng làm đối chứng cho kNN và kiểm tra độ bền vững với ngoại lai.
3. **Lưu trữ dữ liệu sạch:**
   - Xuất toàn bộ dữ liệu sạch và các tập Train/Test đã xử lý ra thư mục `data/` để phục vụ trực tiếp cho giai đoạn huấn luyện mô hình.

In [27]:
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# 1. Tách ma trận đặc trưng X và biến mục tiêu y
# Loại bỏ cnt (mục tiêu) và 2 cột tọa độ 2D pca_1, pca_2 (vốn chỉ phục vụ trực quan hóa ở bước trước)
feature_cols = [c for c in df.columns if c not in ["cnt", "pca_1", "pca_2"]]
X = df[feature_cols].copy()
y = df["cnt"].copy()

# 2. Phân chia tập dữ liệu 80% Train - 20% Test TRƯỚC KHI CHUẨN HÓA
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Khởi tạo và Fit các bộ chuẩn hóa CHỈ TRÊN X_train (Zero Data Leakage)
# Bộ chuẩn hóa chính: StandardScaler
std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train)
X_test_std = std_scaler.transform(X_test)

# Bộ chuẩn hóa đối chứng: MinMaxScaler và RobustScaler
minmax_scaler = MinMaxScaler()
X_train_minmax = minmax_scaler.fit_transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

robust_scaler = RobustScaler()
X_train_robust = robust_scaler.fit_transform(X_train)
X_test_robust = robust_scaler.transform(X_test)

# Chuyển đổi lại DataFrame để giữ nguyên tên cột
X_train_scaled = pd.DataFrame(X_train_std, columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_std, columns=feature_cols, index=X_test.index)

# 4. Xuất các tập dữ liệu ra thư mục data/
data_dir = "../data" if os.path.exists("../data") else "data"

# Dữ liệu sạch tổng hợp 17,379 dòng
df_export = df[feature_cols + ["cnt"]].copy()
df_export.to_csv(os.path.join(data_dir, "hour_cleaned.csv"), index=False)

# Tập Train / Test gốc (chưa scale - dùng cho các mô hình Cây như Decision Tree, Random Forest, XGBoost)
X_train.to_csv(os.path.join(data_dir, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(data_dir, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(data_dir, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(data_dir, "y_test.csv"), index=False)

# Tập Train / Test đã chuẩn hóa StandardScaler (dùng cho Linear Regression, Ridge, Lasso, SVR, kNN, MLP)
X_train_scaled.to_csv(os.path.join(data_dir, "X_train_scaled.csv"), index=False)
X_test_scaled.to_csv(os.path.join(data_dir, "X_test_scaled.csv"), index=False)

print("=== THÔNG SỐ PHÂN CHIA TẬP DỮ LIỆU (TRAIN / TEST) ===")
print(f"Tập huấn luyện (X_train): {X_train.shape[0]} dòng ({X_train.shape[0]/len(df)*100:.1f}%), {X_train.shape[1]} đặc trưng")
print(f"Tập kiểm tra (X_test):     {X_test.shape[0]} dòng ({X_test.shape[0]/len(df)*100:.1f}%), {X_test.shape[1]} đặc trưng")
print(f"Biến mục tiêu y_train:     {y_train.shape[0]} dòng")
print(f"Biến mục tiêu y_test:      {y_test.shape[0]} dòng")
print("\n=== KIỂM CHỨNG CHỐNG RÒ RỈ DỮ LIỆU (ZERO DATA LEAKAGE) ===")
print(f"X_train_scaled -> Mean: {X_train_scaled.mean().mean():.4f}, Std: {X_train_scaled.std().mean():.4f}")
print(f"X_test_scaled  -> Mean: {X_test_scaled.mean().mean():.4f}, Std: {X_test_scaled.std().mean():.4f}")
print(f"\nDanh sách 23 đặc trưng sẵn sàng cho huấn luyện:\n{list(feature_cols)}")

=== THÔNG SỐ PHÂN CHIA TẬP DỮ LIỆU (TRAIN / TEST) ===
Tập huấn luyện (X_train): 13903 dòng (80.0%), 23 đặc trưng
Tập kiểm tra (X_test):     3476 dòng (20.0%), 23 đặc trưng
Biến mục tiêu y_train:     13903 dòng
Biến mục tiêu y_test:      3476 dòng

=== KIỂM CHỨNG CHỐNG RÒ RỈ DỮ LIỆU (ZERO DATA LEAKAGE) ===
X_train_scaled -> Mean: 0.0000, Std: 1.0000
X_test_scaled  -> Mean: 0.0012, Std: 1.0164

Danh sách 23 đặc trưng sẵn sàng cho huấn luyện:
['yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'temp', 'hum', 'windspeed', 'hr_sin', 'hr_cos', 'mnth_sin', 'mnth_cos', 'weekday_sin', 'weekday_cos', 'is_rush_hour', 'season_2', 'season_3', 'season_4', 'weathersit_2', 'weathersit_3', 'weathersit_4', 'hour_cluster']


**Tổng kết giai đoạn Tiền xử lý dữ liệu và Kỹ thuật đặc trưng:**
- Nhóm em đã hoàn tất toàn bộ quy trình tiền xử lý dữ liệu theo chuẩn mực Học máy cổ điển:
  1. Loại bỏ các thuộc tính gây nhiễu, rò rỉ dữ liệu (`instant`, `dteday`, `casual`, `registered`) và đa cộng tuyến (`atemp`).
  2. Xử lý triệt để 22 dòng lỗi cảm biến độ ẩm (`hum == 0`) bằng giá trị trung vị theo nhóm tháng-giờ.
  3. Lọc ngoại lai biến mục tiêu `cnt` bằng phương pháp Capping (Winsorization) theo IQR từng mùa để bảo vệ các mô hình hồi quy mà không làm mất dữ liệu.
  4. Tạo 6 đặc trưng chu kỳ lượng giác $\sin/\cos$ cho giờ, tháng, thứ; tạo biến nhị phân `is_rush_hour`; mã hóa One-Hot cho các biến phân loại `season`, `weathersit` (`drop_first=True`).
  5. Phân cụm K-Means cho 24 khung giờ tạo đặc trưng mới `hour_cluster` và trực quan hóa cấu trúc không gian đặc trưng bằng PCA.
  6. Phân chia Train/Test 80–20 và chuẩn hóa dữ liệu tuân thủ nghiêm ngặt nguyên tắc Zero Data Leakage.
- Toàn bộ các tập dữ liệu sạch đã được lưu trữ sẵn sàng trong thư mục `data/` để bước vào **Giai đoạn 5: Huấn luyện, Tinh chỉnh và Đánh giá các mô hình học máy**.